# SixDegrees — AI Profile Similarity Demo

SixDegrees now uses **sentence-transformer embeddings** to measure how similar two users are — going beyond simple keyword matching.

### The problem with keyword matching
Old approach: Jaccard similarity on interest tokens.  
> `"hiking"` vs `"trail running"` → **0% similar** (no shared tokens)

### The new approach
We embed each user's interests + bio into a 384-dimensional vector using [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), then measure **cosine similarity** between vectors.  
> `"hiking"` vs `"trail running"` → **~54% similar** (the model understands they're related)

This notebook pulls **real user profiles from the database** and shows the difference live.

In [1]:
import os
import sys
from dotenv import load_dotenv

load_dotenv('../backend/.env')
sys.path.insert(0, '../backend')

from config.settings import get_supabase_client
from models.user import UserProfile
from services.matching.embedder import build_profile_text, embed_profiles, cosine_sim
from services.matching.similarity import jaccard

print('Connected to Supabase ✓')

Connected to Supabase ✓


## Step 1 — Fetch real profiles from the database

In [2]:
sb = get_supabase_client()

# Fetch profiles that have at least some interests or a bio
rows = sb.table('profiles').select(
    'id, nickname, interests, bio, city, state, age, industry'
).execute().data

# Keep only profiles with meaningful text content
profiles = [
    UserProfile(**r) for r in rows
    if r.get('interests') or r.get('bio')
]

print(f'Loaded {len(profiles)} profiles with interests or bio')

Loaded 168 profiles with interests or bio


## Step 2 — Look at a few profiles

These are real users. The text we embed for each person is their **interests joined together + bio**.

In [3]:
# Show the first 6 profiles and the text that gets embedded
sample = profiles[:6]

for p in sample:
    interests_str = ', '.join(p.interests) if p.interests else '—'
    bio_str = p.bio or '—'
    embed_text = build_profile_text(p) or '(empty)'
    location = f'{p.city}, {p.state}' if p.city else '—'
    print(f'  @{p.nickname:<18} | interests: {interests_str}')
    print(f'  {"":<20} | bio: {bio_str}')
    print(f'  {"":<20} | → embedded as: "{embed_text}"')
    print()

  @OnceMore           | interests: to
                       | bio: —
                       | → embedded as: "to"

  @Ell13              | interests: Coding, Basketball, Jazz
                       | bio: —
                       | → embedded as: "Coding Basketball Jazz"

  @Tina               | interests: birdwatching, sewing, roller skating
                       | bio: —
                       | → embedded as: "birdwatching sewing roller skating"

  @Div                | interests: Running, crochet
                       | bio: I am crashing out. What even is molecular biology :(
                       | → embedded as: "Running crochet. I am crashing out. What even is molecular biology :("

  @Auto2              | interests: Jumping
                       | bio: I'm cool
                       | → embedded as: "Jumping. I'm cool"

  @Eleanor            | interests: art, music, film
                       | bio: —
                       | → embedded as: "art music film"



## Step 3 — Embed all profiles in one batch

We call `embed_profiles()` once — it runs one forward pass through the model for all users simultaneously.  
Each profile becomes a single **384-dimensional vector**.

In [4]:
import numpy as np

embeddings_matrix = embed_profiles(profiles)  # shape: (N, 384)
embeddings = {p.id: embeddings_matrix[i] for i, p in enumerate(profiles)}

print(f'Embedding matrix shape: {embeddings_matrix.shape}')
print(f'Each profile → 384-dimensional vector')
print(f'Example vector (first 8 dims of @{profiles[0].nickname}): {embeddings_matrix[0, :8].round(4)}')

Embedding matrix shape: (168, 384)
Each profile → 384-dimensional vector
Example vector (first 8 dims of @OnceMore): [-0.022   0.0429 -0.0413  0.0804 -0.0157  0.0347  0.1349 -0.034 ]


## Step 4 — Compare similarity between real users

For each pair of the first 6 users: compute **Jaccard** (old) vs **cosine similarity** (new AI).  
The AI score captures semantic meaning — similar topics score higher even without shared keywords.

In [5]:
print(f'{"User A":<18}  {"User B":<18}  {"Jaccard":>8}  {"AI cos-sim":>10}  {"Delta":>8}')
print('-' * 72)

for i in range(len(sample)):
    for j in range(i + 1, len(sample)):
        a, b = sample[i], sample[j]
        j_score = jaccard(a.interests, b.interests, stem=True)
        ai_score = cosine_sim(embeddings[a.id], embeddings[b.id])
        delta = ai_score - j_score
        marker = '  ← AI finds hidden similarity!' if delta > 0.15 else ''
        print(f'@{a.nickname:<17}  @{b.nickname:<17}  {j_score:>8.3f}  {ai_score:>10.3f}  {delta:>+8.3f}{marker}')

User A              User B               Jaccard  AI cos-sim     Delta
------------------------------------------------------------------------
@OnceMore           @Ell13                 0.000       0.166    +0.166  ← AI finds hidden similarity!
@OnceMore           @Tina                  0.000       0.065    +0.065
@OnceMore           @Div                   0.000       0.098    +0.098
@OnceMore           @Auto2                 0.000       0.217    +0.217  ← AI finds hidden similarity!
@OnceMore           @Eleanor               0.000       0.086    +0.086
@Ell13              @Tina                  0.000       0.275    +0.275  ← AI finds hidden similarity!
@Ell13              @Div                   0.000       0.109    +0.109
@Ell13              @Auto2                 0.000       0.213    +0.213  ← AI finds hidden similarity!
@Ell13              @Eleanor               0.000       0.231    +0.231  ← AI finds hidden similarity!
@Tina               @Div                   0.000       0.230  

## Step 5 — Find the top match for a specific user

Pick any user from the database and rank everyone else by **AI similarity score**.

In [6]:
# Pick the first profile as our subject
subject = profiles[0]
others = profiles[1:]

subject_vec = embeddings[subject.id]

scores = []
for other in others:
    ai_score = cosine_sim(subject_vec, embeddings[other.id])
    j_score = jaccard(subject.interests, other.interests, stem=True)
    scores.append((other, ai_score, j_score))

scores.sort(key=lambda x: x[1], reverse=True)
top = scores[:8]

subject_text = build_profile_text(subject) or '(empty)'
print(f'Subject: @{subject.nickname}')
print(f'Profile text: "{subject_text}"')
print()
print(f'Top matches by AI similarity:')
print(f'{"Rank":<6} {"Nickname":<18} {"AI score":>9} {"Jaccard":>8}  Profile text')
print('-' * 78)
for rank, (other, ai_score, j_score) in enumerate(top, 1):
    text = build_profile_text(other) or '(empty)'
    print(f'{rank:<6} @{other.nickname:<17} {ai_score:>9.3f} {j_score:>8.3f}  "{text[:45]}"')

Subject: @OnceMore
Profile text: "to"

Top matches by AI similarity:
Rank   Nickname            AI score  Jaccard  Profile text
------------------------------------------------------------------------------
1      @wispy                 0.396    0.000  "Winning"
2      @Hi                    0.332    0.000  "sleep"
3      @user_090              0.327    0.000  "science"
4      @user_036              0.321    0.000  "gaming"
5      @user_074              0.321    0.000  "gaming"
6      @first                 0.320    0.000  "C"
7      @user_019              0.298    0.000  "politics"
8      @user_083              0.270    0.000  "finance"


## Summary

| | Jaccard (old) | AI Embeddings (new) |
|---|---|---|
| **How it works** | Count shared interest tokens | Encode meaning into 384-dim vectors, measure angle |
| **`hiking` vs `trail running`** | 0% (no shared tokens) | ~54% (semantically related) |
| **`guitar` vs `music`** | 0% | high similarity |
| **Model** | — | `all-MiniLM-L6-v2`, runs locally, ~90MB |
| **What's embedded** | — | `interests` + `bio` joined into one string |
| **Weight in final score** | 40% | 40% (same slot, drop-in replacement) |

The rest of the score (location, languages, education, industry, age) still uses hand-crafted functions — no change there.